In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------
# 1. CARREGAMENTO E PREPARAÇÃO DOS DADOS
# ---------------------------------------------------------
# ATENÇÃO: Se o seu DataFrame já está carregado na memória como 'df', 
# você pode comentar ou apagar a linha abaixo.
df = pd.read_csv('df_recife.csv')

# Converte as colunas 'Data' e 'Hora UTC' para o formato datetime, se elas existirem
if 'Data' in df.columns and 'Hora UTC' in df.columns:
    string_temporaria = df['Data'] + ' ' + df['Hora UTC']
    df['datahora'] = pd.to_datetime(string_temporaria, format='%Y/%m/%d %H%M UTC')
    # Remove as colunas antigas para limpar o dataframe
    df = df.drop(columns=['Data', 'Hora UTC'])

# Ordenar por data para garantir a sequência temporal exata (vital para o T-6h)
df = df.sort_values('datahora').reset_index(drop=True)

# ---------------------------------------------------------
# 2. ABORDAGEM 1: DISTRIBUIÇÃO GERAL (BOXPLOT)
# ---------------------------------------------------------
def classificar_intensidade(prec):
    if prec > 10.0:
        return 'Intensa (> 10 mm/h)'
    elif prec > 2.0 and prec <= 10.0:
        return 'Moderada'
    elif prec > 0 and prec <= 2.0:
        return 'Fraca (<= 2 mm/h)'
    else:
        return 'Sem Chuva'

# Cria a coluna de classificação baseada na precipitação
df['classe_chuva'] = df['precipitacao'].apply(classificar_intensidade)

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df, 
    x='classe_chuva', 
    y='temperatura',
    order=['Sem Chuva', 'Fraca (<= 2 mm/h)', 'Moderada', 'Intensa (> 10 mm/h)'],
    palette='Blues'
)
plt.title('Distribuição da Temperatura por Intensidade de Chuva em Recife', fontsize=14)
plt.xlabel('Intensidade da Chuva', fontsize=12)
plt.ylabel('Temperatura (°C)', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 3. ABORDAGEM 2: EVOLUÇÃO TEMPORAL (T-6h a T-0)
# ---------------------------------------------------------
# Encontrar os índices onde ocorreu chuva intensa e chuva fraca
indices_chuva_intensa = df[df['precipitacao'] > 10.0].index
indices_chuva_fraca = df[(df['precipitacao'] > 0) & (df['precipitacao'] <= 2.0)].index

def extrair_janela_t6(df, indices, label):
    dados_janela = []
    for idx in indices:
        # Verifica se temos 6 horas de histórico para trás
        if idx >= 6: 
            janela = df.loc[idx-6 : idx, 'temperatura'].values
            # Se não houver dados nulos na janela e o tamanho for 7 (T-6 até T-0)
            if len(janela) == 7 and not pd.isna(janela).any():
                dados_janela.append(janela)
    
    # Cria um DataFrame com as 6 horas anteriores + a hora do evento
    df_janela = pd.DataFrame(dados_janela, columns=['T-6h', 'T-5h', 'T-4h', 'T-3h', 'T-2h', 'T-1h', 'T-0'])
    # Calcula a média de temperatura para cada hora
    media_janela = df_janela.mean()
    
    return pd.DataFrame({'Hora': media_janela.index, 'Temperatura Média': media_janela.values, 'Categoria': label})

# Extraindo as médias
df_evolucao_intensa = extrair_janela_t6(df, indices_chuva_intensa, 'Antes de Chuva Intensa (> 10 mm/h)')
df_evolucao_fraca = extrair_janela_t6(df, indices_chuva_fraca, 'Antes de Chuva Fraca (<= 2 mm/h)')

# Unindo para plotar
df_plot_linhas = pd.concat([df_evolucao_intensa, df_evolucao_fraca])

plt.figure(figsize=(10, 6))
sns.lineplot(
    data=df_plot_linhas, 
    x='Hora', 
    y='Temperatura Média', 
    hue='Categoria', 
    marker='o', 
    linewidth=2.5,
    palette=['#005b96', '#6497b1'] # Cores alinhadas com o padrão do projeto
)

plt.title('Evolução da Temperatura Média Antes da Chuva (T-6h a T-0)', fontsize=14)
plt.xlabel('Tempo antes do evento de chuva', fontsize=12)
plt.ylabel('Temperatura Média (°C)', fontsize=12)
plt.legend(title='')
plt.grid(True, linestyle='--', alpha=0.5)

# Mapeamento seguro das posições do Eixo X (Corrige o ConversionError)
posicoes_x = {'T-6h': 0, 'T-5h': 1, 'T-4h': 2, 'T-3h': 3, 'T-2h': 4, 'T-1h': 5, 'T-0': 6}

# Adicionando os valores numéricos usando a posição exata (0, 1, 2...)
for linha in df_plot_linhas.itertuples():
    x_pos = posicoes_x[linha.Hora] 
    
    plt.text(
        x=x_pos, 
        y=linha._2 + 0.15, # 'linha._2' é a Temperatura Média. O +0.15 eleva o número um pouco acima da linha
        s=f'{linha._2:.1f}°', 
        ha='center', 
        fontsize=10, 
        color='black'
    )

plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'df_recife.csv'